# Transformer for Multimodal Stock Forecasting
Implementation notebook for *Adaptive Portfolio Management in Volatile Markets with Multiplex Attention Transformers and Deep Reinforcement Learning*.

Code for **PyTorch** implementation of the Transformer proposed in our paper, trained to predict multi-step stock price trajectories from:
1. **Technical indicators** (OHLCV-derived)
2. **News‑sentiment embeddings** (from FinBERT)
3. **Macroeconomic features**

The model outputs \(H\)-step forecasts that later serve as inputs to the downstream RL trading agent.

In [1]:
!pip install -U torch torchvision torchaudio --quiet
!pip install -U transformers datasets peft --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [2]:
import math, os, random, json, itertools, warnings, datetime as dt, typing as typ
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on', device)

Running on cuda


In [61]:
# some hyperparameters
WINDOW_SIZE = 30  # T
HORIZON_SIZE = 1  # H
BATCH_SIZE = 64
NUM_EPOCHS = 50

# feature dimensions
TECH_DIM = 10
SENT_DIM = 1
MACRO_DIM = 3

## Data Processing

In [72]:
import pandas as pd, numpy as np
import pandas as pd, numpy as np, torch
from torch.utils.data import Dataset, DataLoader


def load_tensor(csv_path: str, feature_cols: list[str],
                tickers=None, dates=None):
    df = pd.read_csv(csv_path, parse_dates=['date'])

    if tickers is None: tickers = sorted(df['ticker'].unique())
    if dates   is None: dates   = sorted(df['date'].unique())

    df = (df.query('ticker in @tickers and date in @dates')
            .sort_values(['date', 'ticker']))

    mats = []
    for feat in feature_cols:
        mat = (df.pivot(index='date', columns='ticker', values=feat)
                 .reindex(index=dates, columns=tickers)
                 .astype('float32')
                 .fillna(0.0)   # replace missing with 0, try avoiding nan
                 .values)
        mats.append(mat[..., None])

    return np.concatenate(mats, axis=2), tickers, dates

class MultiCSVWindowDataset(Dataset):
    """
    Each sample = (x_stock, x_sent, x_macro, y)
    Shapes:
      x_* : [window, F]   (one ticker, consecutive dates)
      y   : [horizon, 4]  (open/close/high/low for next horizon days)
    """
    def __init__(self, npz_path: str, window: int = WINDOW_SIZE, horizon: int = HORIZON_SIZE):
        data = np.load(npz_path)
        self.stock  = data['tech']    # [D, N, F1]
        self.sent   = data['sent']    # [D, N, F2]
        self.macro  = data['macro']   # [D, N, F3]
        self.labels = data['price']   # [D, N, 4]

        self.T = window
        self.H = horizon
        self.D = self.stock.shape[0] - window - horizon
        self.N = self.stock.shape[1]

    def __len__(self):
        return self.D * self.N            # all (ticker, start‑time) pairs

    def __getitem__(self, idx):
        t0 = idx % self.D                 # time start
        k  = idx // self.D                # ticker index

        x_stock = self.stock [t0:t0+self.T, k, :]
        x_sent  = self.sent  [t0:t0+self.T, k, :]
        x_macro = self.macro[t0:t0+self.T, k, :]
        y = self.labels[t0+self.T:t0+self.T+self.H, k, :]   # horizon×4

        # previous day's true close:
        # this is for calculating accuracy only
        prev_close = self.labels[t0+self.T-1, k, 1]     # scalar

        return (torch.tensor(x_stock),
                torch.tensor(x_sent),
                torch.tensor(x_macro),
                torch.tensor(y),
                torch.tensor(prev_close))

# feature scaling (min‑max to 0‑1)
def minmax_scale(arr, eps=1e-8):
    mn = np.nanmin(arr, axis=(0,1), keepdims=True)
    mx = np.nanmax(arr, axis=(0,1), keepdims=True)
    return (arr - mn) / (mx - mn + eps), mn, mx   # eps avoids /0 → NaN



In [74]:
# get train data
# CONFIG
TECH_CSV  = "train_stock.csv"      #  must contain: date, ticker, tech columns
SENT_CSV  = "train_sentiment.csv"      #  must contain: date, ticker, sentiment cols
MACRO_CSV = "train_macro.csv"     #  must contain: date, ticker, macro cols
LABEL_CSV = "train_stock.csv"  # contains: date, ticker, open, close, high, low

TECH_COLS  = ['volume','macd','boll_ub','boll_lb','rsi_30','cci_30',
              'dx_30','close_30_sma','close_60_sma','vix']
SENT_COLS  = ['sentiment']
MACRO_COLS = ['fedfunds','sp500','sector']
LABEL_COLS = ['open','close','high','low']


# build aligned tensors
tech, tickers, dates  = load_tensor(TECH_CSV,  TECH_COLS)
sent, _, _ = load_tensor(SENT_CSV,  SENT_COLS,  tickers, dates)
macro, _, _  = load_tensor(MACRO_CSV, MACRO_COLS, tickers, dates)

#  build 4‑price cube (Date × Ticker × 4)
lbl_df = pd.read_csv(LABEL_CSV, parse_dates=['date'])
lbl_df = (
    lbl_df.query('ticker in @tickers and date in @dates')
          .sort_values(['date', 'ticker'])
)

label_mats = []
for col in LABEL_COLS:         # ['open','close','high','low']
    mat = (lbl_df.pivot(index='date', columns='ticker', values=col)
                  .reindex(index=dates, columns=tickers)     # enforce exact order
                  .astype('float32')
                  .values)                                   # (D, N)
    label_mats.append(mat[..., None])                        # add 1‑len feature axis

labels = np.concatenate(label_mats, axis=2)         # (D, N, 4)


tech_scaled,  tech_min,  tech_max  = minmax_scale(tech)
sent_scaled,  sent_min,  sent_max  = minmax_scale(sent)
macro_scaled, macro_min, macro_max = minmax_scale(macro)

labels_scaled, price_min, price_max = minmax_scale(labels)

np.savez_compressed(
    "aligned_data_train.npz",
    tech=tech_scaled, sent=sent_scaled,
    macro=macro_scaled, price=labels)


dataset = MultiCSVWindowDataset("aligned_data_train.npz", window=WINDOW_SIZE, horizon=HORIZON_SIZE)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# for batch in loader:
#     x_stock_train, x_sent_train, x_macro_train, y_train, = batch   # ready for training step
#     # print(batch)
#     break


# right after building train_dataset, check shape
x_s, x_se, x_m, y, close = dataset[0]
print("First train window shapes:", x_s.shape, x_se.shape, x_m.shape, y.shape)
print(" First target (raw):", (y * (price_max-price_min) + price_min).round())
print(" First features peek:", x_s[-1, :3].round())  # last time‐step stock features



First train window shapes: torch.Size([30, 10]) torch.Size([30, 1]) torch.Size([30, 3]) torch.Size([1, 4])
 First target (raw): tensor([[[1536., 1220., 1539., 1476.]]])
 First features peek: tensor([0., 1., 0.])


In [75]:
# get validation data
# CONFIG
TECH_CSV  = "val_stock.csv"      #  must contain: date, ticker, tech columns
SENT_CSV  = "val_sentiment.csv"      #  must contain: date, ticker, sentiment cols
MACRO_CSV = "val_macro.csv"     #  must contain: date, ticker, macro cols
LABEL_CSV = "val_stock.csv"  # contains: date, ticker, open, close, high, low

TECH_COLS  = ['volume','macd','boll_ub','boll_lb','rsi_30','cci_30',
              'dx_30','close_30_sma','close_60_sma','vix']
SENT_COLS  = ['sentiment']
MACRO_COLS = ['fedfunds','sp500','sector']
LABEL_COLS = ['open','close','high','low']


# build aligned tensors
tech, tickers, dates  = load_tensor(TECH_CSV,  TECH_COLS)
sent, _, _ = load_tensor(SENT_CSV,  SENT_COLS,  tickers, dates)
macro, _, _  = load_tensor(MACRO_CSV, MACRO_COLS, tickers, dates)

#  build 4‑price cube (Date × Ticker × 4)
lbl_df = pd.read_csv(LABEL_CSV, parse_dates=['date'])
lbl_df = (
    lbl_df.query('ticker in @tickers and date in @dates')
          .sort_values(['date', 'ticker'])
)

label_mats = []
for col in LABEL_COLS:         # ['open','close','high','low']
    mat = (lbl_df.pivot(index='date', columns='ticker', values=col)
                  .reindex(index=dates, columns=tickers)     # enforce exact order
                  .astype('float32')
                  .values)                                   # (D, N)
    label_mats.append(mat[..., None])                        # add 1‑len feature axis

labels = np.concatenate(label_mats, axis=2)         # (D, N, 4)


tech_scaled,  tech_min,  tech_max  = minmax_scale(tech)
sent_scaled,  sent_min,  sent_max  = minmax_scale(sent)
macro_scaled, macro_min, macro_max = minmax_scale(macro)

labels_scaled, price_min, price_max = minmax_scale(labels)

np.savez_compressed(
    "aligned_data_val.npz",
    tech=tech_scaled, sent=sent_scaled,
    macro=macro_scaled, price=labels)

dataset = MultiCSVWindowDataset("aligned_data_val.npz", window=WINDOW_SIZE, horizon=HORIZON_SIZE)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# for batch in loader:
#     x_stock_val, x_sent_val, x_macro_val, y_val = batch
#     # print(batch)
#     break



In [76]:
# get test data
# CONFIG
TECH_CSV  = "test_stock.csv"      #  must contain: date, ticker, tech columns
SENT_CSV  = "test_sentiment.csv"      #  must contain: date, ticker, sentiment cols
MACRO_CSV = "test_macro.csv"     #  must contain: date, ticker, macro cols
LABEL_CSV = "test_stock.csv"  # contains: date, ticker, open, close, high, low

TECH_COLS  = ['volume','macd','boll_ub','boll_lb','rsi_30','cci_30',
              'dx_30','close_30_sma','close_60_sma','vix']
SENT_COLS  = ['sentiment']
MACRO_COLS = ['fedfunds','sp500','sector']
LABEL_COLS = ['open','close','high','low']


# build aligned tensors
tech, tickers, dates  = load_tensor(TECH_CSV,  TECH_COLS)
sent, _, _ = load_tensor(SENT_CSV,  SENT_COLS,  tickers, dates)
macro, _, _  = load_tensor(MACRO_CSV, MACRO_COLS, tickers, dates)

#  build 4‑price cube (Date × Ticker × 4)
lbl_df = pd.read_csv(LABEL_CSV, parse_dates=['date'])
lbl_df = (
    lbl_df.query('ticker in @tickers and date in @dates')
          .sort_values(['date', 'ticker'])
)

label_mats = []
for col in LABEL_COLS:         # ['open','close','high','low']
    mat = (lbl_df.pivot(index='date', columns='ticker', values=col)
                  .reindex(index=dates, columns=tickers)     # enforce exact order
                  .astype('float32')
                  .values)                                   # (D, N)
    label_mats.append(mat[..., None])                        # add 1‑len feature axis

labels = np.concatenate(label_mats, axis=2)         # (D, N, 4)


tech_scaled,  tech_min,  tech_max  = minmax_scale(tech)
sent_scaled,  sent_min,  sent_max  = minmax_scale(sent)
macro_scaled, macro_min, macro_max = minmax_scale(macro)

labels_scaled, price_min, price_max = minmax_scale(labels)

np.savez_compressed(
    "aligned_data_test.npz",
    tech=tech_scaled, sent=sent_scaled,
    macro=macro_scaled, price=labels)

dataset = MultiCSVWindowDataset("aligned_data_test.npz", window=WINDOW_SIZE, horizon=HORIZON_SIZE)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


# for batch in loader:
#     x_stock_test, x_sent_test, x_macro_test, y_test = batch
#     # print(batch)
#     break



## Multiplex Attention Block

In [77]:

class MultiplexAttention(nn.Module):
    """Applies self‑attention independently to three modality streams and concatenates the results."""

    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.attn_stock = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_sent  = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_macro = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        # projection after concat of 3*dim back to dim
        self.out_proj   = nn.Linear(3*dim, dim)
        #self.norm = nn.LayerNorm(dim)



    def forward(self, x_stock, x_sent, x_macro, key_padding_mask=None):
        seq_len = x_stock.size(1)
        attn_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1).to(x_stock.device)

        # each input: [B, T, D]
        a_stock, _ = self.attn_stock(x_stock, x_stock, x_stock,
                                     key_padding_mask=key_padding_mask,
                                     attn_mask=attn_mask)
        a_sent,  _ = self.attn_sent(x_sent, x_sent, x_sent,
                                    key_padding_mask=key_padding_mask,
                                    attn_mask=attn_mask)
        a_macro, _ = self.attn_macro(x_macro, x_macro, x_macro,
                                     key_padding_mask=key_padding_mask,
                                     attn_mask=attn_mask)
        # concat on last dim
        cat = torch.cat([a_stock, a_sent, a_macro], dim=-1)
        out = self.out_proj(cat)
        #return self.norm(out)
        return out


## Transformer Forecasting Model

In [78]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class MultiplexTransformerForecaster(nn.Module):
    def __init__(self,
                 dim: int = 128,
                 num_heads: int = 4,
                 num_layers: int = 6,
                 horizon: int = HORIZON_SIZE,
                 dropout: float = 0.1,
                 tech_dim:  int = TECH_DIM,
                 sent_dim:  int = SENT_DIM,
                 macro_dim: int = MACRO_DIM,
                 window:    int = WINDOW_SIZE):
        super().__init__()
        self.horizon = horizon
        self.window  = window
        self.attn_norm = nn.LayerNorm(dim)

        # modality‑specific projections
        self.embed_stock = nn.Linear(tech_dim,  dim)
        self.embed_sent  = nn.Linear(sent_dim,  dim)
        self.embed_macro = nn.Linear(macro_dim, dim)

        self.pos_enc = PositionalEncoding(dim)

        blocks = []
        for _ in range(num_layers):
            blocks.append(MultiplexAttention(dim, num_heads))
            blocks.append(nn.Sequential(
                nn.LayerNorm(dim),
                nn.Linear(dim, dim*4), nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(dim*4, dim)
            ))
        self.layers = nn.ModuleList(blocks)

        # prediction head: produce horizon × 4 prices, not just horizon
        self.pred_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Flatten(start_dim=1),                      # [B, window*dim]
            nn.Linear(dim * window, horizon * 4)
        )

    def forward(self, x_stock, x_sent, x_macro):          # [B, T, F*]
        s = self.embed_stock(x_stock)
        t = self.embed_sent(x_sent)
        m = self.embed_macro(x_macro)

        s, t, m = map(self.pos_enc, (s, t, m))

        for blk in self.layers:
            if isinstance(blk, MultiplexAttention):
                residual = s
                s = t = m = blk(s, t, m)
                s = s + residual
                s = self.attn_norm(s)
            else:                         # feed‑forward
                s = s + blk(s)
                t = m = s

        out = self.pred_head(s)                           # [B, horizon*4]
        return out.view(-1, self.horizon, 4)              # [B, H, 4]


### Inject LoRA Adapters

In [41]:

# Uncomment to add LoRA adapters if `peft` is installed in environment
# from peft import get_peft_model, LoraConfig, TaskType
# lora_cfg = LoraConfig(
#     task_type=TaskType.FEATURE_EXTRACTION,
#     target_modules=['q_proj', 'k_proj', 'v_proj'],
#     r=8, lora_alpha=32, lora_dropout=0.05)
# model = MultiplexTransformerForecaster().to(device)
# model = get_peft_model(model, lora_cfg)


## Training Loop

In [81]:

def train_epoch(model, loader, optimizer, scaler, loss_fn, grad_clip=1.0):
    model.train()
    total_loss = 0
    tp = tn = fp = fn = 0

    # broadcast min/max into torch tensors
    pm = torch.tensor(price_min, dtype=torch.float32, device=device)
    pM = torch.tensor(price_max, dtype=torch.float32, device=device)


    for x_stock, x_sent, x_macro, y, prev_close in loader:
        x_stock = x_stock.to(device)
        x_sent  = x_sent.to(device)
        x_macro = x_macro.to(device)
        y = y.to(device)
        prev_close = prev_close.to(device)

        # for debugging
        #print(y)

        # prediction
        y_hat = model(x_stock, x_sent, x_macro)
        loss = loss_fn(y_hat, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * y.size(0)
        #print(loss.item())

        pred_up = (y_hat[:, 0, 1] > prev_close)
        true_up = (y[:, 0, 1] > prev_close)

        tp += ((pred_up == 1) & (true_up == 1)).sum().item()
        tn += ((pred_up == 0) & (true_up == 0)).sum().item()
        fp += ((pred_up == 1) & (true_up == 0)).sum().item()
        fn += ((pred_up == 0) & (true_up == 1)).sum().item()

    acc = (tp + tn) / (tp + tn + fp + fn) if (tp+tn+fp+fn) > 0 else 0.0
    return total_loss / len(loader.dataset), acc

def evaluate_epoch(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_count = 0
    total_loss = 0
    tp = tn = fp = fn = 0

    # broadcast min/max into torch tensors
    pm = torch.tensor(price_min, dtype=torch.float32, device=device)
    pM = torch.tensor(price_max, dtype=torch.float32, device=device)


    with torch.no_grad():
        for x_stock, x_sent, x_macro, y, prev_close in loader:
            x_stock = x_stock.to(device)
            x_sent  = x_sent.to(device)
            x_macro = x_macro.to(device)
            y       = y.to(device)
            prev_close = prev_close.to(device)

            # prediction
            y_hat   = model(x_stock, x_sent, x_macro)
            loss    = loss_fn(y_hat, y)
            batch_n = y.size(0)
            total_loss += loss.item() * batch_n
            total_count += batch_n

            pred_up = (y_hat[:, 0, 1] > prev_close)
            true_up = (y[:, 0, 1] > prev_close)

            tp += ((pred_up == 1) & (true_up == 1)).sum().item()
            tn += ((pred_up == 0) & (true_up == 0)).sum().item()
            fp += ((pred_up == 1) & (true_up == 0)).sum().item()
            fn += ((pred_up == 0) & (true_up == 1)).sum().item()

    acc = (tp + tn) / (tp + tn + fp + fn) if (tp+tn+fp+fn) > 0 else 0.0
    return total_loss / total_count, acc



In [82]:

# training script (adjust paths & hyperparams)
dataset = MultiCSVWindowDataset('aligned_data_train.npz',
                                window=WINDOW_SIZE,
                                horizon=HORIZON_SIZE)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)
model = MultiplexTransformerForecaster(
            tech_dim=TECH_DIM,
            sent_dim=SENT_DIM,
            macro_dim=MACRO_DIM,
            window=WINDOW_SIZE,
            horizon=HORIZON_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-2)
loss_fn = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()


# Load validation dataset
val_dataset = MultiCSVWindowDataset("aligned_data_val.npz", window=WINDOW_SIZE, horizon=HORIZON_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# before you train at all, check “baseline” loss:
with torch.no_grad():
    zero_preds = torch.zeros_like(next(iter(val_loader))[3])   # y shape
    y = next(iter(val_loader))[3]
    baseline_mse = nn.MSELoss()(zero_preds.to(device), y.to(device))
print("Baseline (zero) MSE on val set:", baseline_mse.item())


for epoch in range(50):
    train_loss, train_acc = train_epoch(model, loader, optimizer, scaler, loss_fn)
    val_loss, val_acc = evaluate_epoch(model, val_loader, loss_fn)
    print(f'Epoch {epoch+1}: train MSE = {train_loss:.4f}, val MSE = {val_loss:.4f}, train acc = {train_acc:.4f}, val acc = {val_acc:.4f}')


Baseline (zero) MSE on val set: 13539.265625
Epoch 1: train MSE = 288.3742, val MSE = 2198.7623, train acc = 0.5034, val acc = 0.4852
Epoch 2: train MSE = 72.2493, val MSE = 3934.6985, train acc = 0.5061, val acc = 0.4852
Epoch 3: train MSE = 49.5854, val MSE = 3791.8881, train acc = 0.5057, val acc = 0.4852
Epoch 4: train MSE = 33.4538, val MSE = 3731.3650, train acc = 0.5070, val acc = 0.4852


KeyboardInterrupt: 

# Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Subset
import numpy as np
import torch.nn as nn
import torch.optim as optim

In [ ]:


def cross_validate(dataset, param_grid, folds=3, epochs=10):
    results = []
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)

    for params in param_grid:
        fold_losses = []
        for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
            train_subset = Subset(dataset, train_idx)
            val_subset = Subset(dataset, val_idx)

            train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, drop_last=True)
            val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, drop_last=False)

            model = MultiplexTransformerForecaster(
                tech_dim=TECH_DIM,
                sent_dim=SENT_DIM,
                macro_dim=MACRO_DIM,
                window=WINDOW_SIZE,
                horizon=HORIZON_SIZE,
                **params
            ).to(device)

            optimizer = optim.AdamW(model.parameters(), lr=params.get('lr', 1e-4), weight_decay=1e-2)
            scaler = torch.cuda.amp.GradScaler()
            loss_fn = nn.MSELoss()

            for epoch in range(epochs):
                train_epoch(model, train_loader, optimizer, scaler, loss_fn)

            val_loss = evaluate_epoch(model, val_loader, loss_fn)
            fold_losses.append(val_loss)

        avg_loss = np.mean(fold_losses)
        results.append((params, avg_loss))
        print(f"Params: {params}, Avg Val Loss: {avg_loss:.4f}")

    best_params = min(results, key=lambda x: x[1])[0]
    print(f"\nBest Hyperparameters: {best_params}")
    return best_params, results

In [ ]:
param_grid = [
    {"dropout": 0.1, "dim": 128, "num_heads":4},
    {"dropout": 0.2, "dim": 256, "num_heads":8},
]

In [ ]:
best_params, results = cross_validate(dataset, param_grid, folds=3, epochs=10)

## Save Trained Model

In [ ]:

# torch.save(model.state_dict(), 'multiplex_transformer.pth')


### Next Steps and Integration
- Add an evaluation loop on the validation/test split.
- Export the frozen Transformer to feed the RL environment.
- Experiment with different window/horizon sizes, LoRA ranks, and learning rates.